# Decision Intelligence Assistant

This notebook covers data preparation, weak labeling, feature engineering, model comparison, and artifact export for the project.

## 1. Goals

- Build a representative 10k-row sample from the Twitter support dataset.
- Create a transparent weak-labeling rule for ticket priority.
- Compare TF-IDF, engineered, and combined features.
- Select a deployable ML baseline and export its artifacts.
- Collect examples for the final written analysis.

## 2. Imports

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import re
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
TARGET_SAMPLE_SIZE = 10_000

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_raw_dir = project_root / "data" / "raw"
data_sample_dir = project_root / "data" / "sample"
artifacts_dir = project_root / "artifacts"
backend_path = project_root / "backend"
if str(backend_path) not in sys.path:
    sys.path.append(str(backend_path))

data_sample_dir.mkdir(parents=True, exist_ok=True)
artifacts_dir.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print(f"Project root: {project_root}")
print(f"Raw data dir: {data_raw_dir}")
print(f"Sample output dir: {data_sample_dir}")


Project root: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant
Raw data dir: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\data\raw
Sample output dir: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\data\sample


In [2]:
from pathlib import Path
import shutil

try:
    import kagglehub
except ImportError:
    kagglehub = None

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

if kagglehub is None:
    print("kagglehub is not installed. Place twcs.csv in data/raw manually.")
else:
    download_path = Path(
        kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
    )
    print("Downloaded to:", download_path)

    csv_files = list(download_path.rglob("*.csv"))
    print("CSV files found:", [p.name for p in csv_files])

    preferred = next((p for p in csv_files if p.name == "twcs.csv"), None)
    source_csv = preferred or next((p for p in csv_files if p.name == "sample.csv"), None)
    if source_csv is None:
        raise FileNotFoundError("No CSV file was found in the downloaded dataset.")

    target_csv = raw_dir / source_csv.name
    shutil.copy2(source_csv, target_csv)
    print("Copied dataset to:", target_csv)
    if source_csv.name != "twcs.csv":
        print("Warning: using sample.csv fallback, not the full twcs.csv dataset.")


kagglehub is not installed. Place twcs.csv in data/raw manually.


## 3. Load Raw Dataset

In [3]:
def parse_linked_ids(raw_value: Any) -> list[str]:
    """Parse linked tweet identifiers from a raw cell value."""
    if pd.isna(raw_value):
        return []

    raw_text = str(raw_value).strip()
    if not raw_text:
        return []

    return [token for token in re.split(r"[\s,\[\]]+", raw_text) if token]


def to_bool_flag(raw_value: Any) -> bool:
    """Coerce the inbound column to a real boolean flag."""
    if isinstance(raw_value, bool):
        return raw_value
    if pd.isna(raw_value):
        return False

    normalized = str(raw_value).strip().lower()
    return normalized in {"true", "t", "1", "yes"}


def load_raw_dataset(raw_dir: Path) -> tuple[pd.DataFrame, Path]:
    """Load the first supported dataset found in data/raw."""
    csv_candidates = sorted(raw_dir.glob("*.csv"))
    parquet_candidates = sorted(raw_dir.glob("*.parquet"))
    preferred_csv = [path for path in csv_candidates if path.name == "twcs.csv"]
    fallback_csv = [path for path in csv_candidates if path.name != "twcs.csv"]
    candidates = preferred_csv + fallback_csv + parquet_candidates

    if not candidates:
        raise FileNotFoundError(
            "No raw dataset was found in data/raw. Place the Kaggle Twitter support "
            "CSV or Parquet file there before running this notebook."
        )

    dataset_path = candidates[0]
    if dataset_path.suffix == ".csv":
        dataframe = pd.read_csv(dataset_path)
    else:
        dataframe = pd.read_parquet(dataset_path)

    return dataframe, dataset_path


raw_df, dataset_path = load_raw_dataset(data_raw_dir)
print(f"Loaded dataset: {dataset_path.name}")
print(f"Raw shape: {raw_df.shape}")

required_columns = {"tweet_id", "author_id", "inbound", "text"}
missing_columns = required_columns - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = raw_df.copy()
df["tweet_id"] = df["tweet_id"].astype(str)
df["author_id"] = df["author_id"].astype(str)
df["text"] = df["text"].fillna("").astype(str)
df["inbound_flag"] = df["inbound"].apply(to_bool_flag)

tweet_to_author = df.set_index("tweet_id")["author_id"].to_dict()
if "response_tweet_id" in df.columns:
    df["brand_hint"] = df["response_tweet_id"].apply(
        lambda value: next(
            (tweet_to_author[tweet_id] for tweet_id in parse_linked_ids(value)
             if tweet_id in tweet_to_author),
            "unknown",
        )
    )
else:
    df["brand_hint"] = "unknown"

customer_tickets = df.loc[df["inbound_flag"]].copy()
customer_tickets["text"] = customer_tickets["text"].str.strip()
customer_tickets = customer_tickets.loc[customer_tickets["text"].ne("")].copy()
customer_tickets["normalized_text"] = (
    customer_tickets["text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
customer_tickets = customer_tickets.drop_duplicates(subset=["tweet_id"])
customer_tickets = customer_tickets.drop_duplicates(subset=["normalized_text"])

customer_tickets["text_length"] = customer_tickets["text"].str.len()
customer_tickets["word_count"] = customer_tickets["text"].str.split().str.len()

print(f"Customer-ticket shape after cleaning: {customer_tickets.shape}")
display(customer_tickets.head())


Loaded dataset: sample.csv
Raw shape: (93, 7)
Customer-ticket shape after cleaning: (49, 12)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,inbound_flag,brand_hint,normalized_text,text_length,word_count
0,119237,105834,True,Wed Oct 11 06:55:44 +0000 2017,@AppleSupport causing the reply to be disregarded and the tapped notification under the keyboard is opened😡😡😡,119236,NaN,True,unknown,@applesupport causing the reply to be disregarded and the tapped notification under the keyboard is opened😡😡😡,109,16
2,119239,105835,True,Wed Oct 11 13:00:09 +0000 2017,@76328 I really hope you all change but I'm sure you won't! Because you don't have to!,119238,NaN,True,ChaseSupport,@76328 i really hope you all change but i'm sure you won't! because you don't have to!,86,17
4,119241,105836,True,Tue Oct 10 15:17:21 +0000 2017,@VirginTrains see attached error message. I've tried leaving a voicemail several times in the past week https://t.co/NxVZjlYx1k,119243,119240.0,True,VirginTrains,@virgintrains see attached error message. i've tried leaving a voicemail several times in the past week https://t.co/nxvzjlyx1k,127,17
6,119244,105836,True,Tue Oct 10 15:26:44 +0000 2017,"@VirginTrains yep, I've tried laptop too several times over the past week and again today. I've tried different browsers too",119245,119243.0,True,VirginTrains,"@virgintrains yep, i've tried laptop too several times over the past week and again today. i've tried different browsers too",124,20
8,119242,105836,True,Tue Oct 10 15:09:00 +0000 2017,@VirginTrains I still haven't heard &amp; the number I'm directed to by phone is a dead end &amp; the live chat doesn't work. Can someone call me?,119240,119246.0,True,VirginTrains,@virgintrains i still haven't heard &amp; the number i'm directed to by phone is a dead end &amp; the live chat doesn't work. can someone call me?,146,27


## 4. Build 10k Representative Sample

In [4]:
def build_representative_sample(
    dataframe: pd.DataFrame,
    target_size: int,
    random_state: int = 42,
) -> pd.DataFrame:
    """Create a reproducible ticket sample that preserves brand variety."""
    if dataframe.empty:
        raise ValueError("Cannot sample from an empty dataframe.")

    sample_size = min(target_size, len(dataframe))
    working_df = dataframe.copy()
    working_df["brand_hint"] = working_df["brand_hint"].fillna("unknown")

    group_sizes = working_df["brand_hint"].value_counts()
    exact_allocations = group_sizes / group_sizes.sum() * sample_size
    allocations = np.floor(exact_allocations).astype(int)
    allocations[allocations == 0] = 1

    while allocations.sum() > sample_size:
        largest_group = allocations.idxmax()
        if allocations[largest_group] > 1:
            allocations[largest_group] -= 1
        else:
            break

    while allocations.sum() < sample_size:
        remainders = exact_allocations - allocations
        next_group = remainders.idxmax()
        allocations[next_group] += 1

    samples = []
    for brand_hint, group_df in working_df.groupby("brand_hint", sort=False):
        requested_n = min(allocations.get(brand_hint, 0), len(group_df))
        if requested_n == 0:
            continue
        sampled_group = group_df.sample(n=requested_n, random_state=random_state)
        samples.append(sampled_group)

    sample_df = pd.concat(samples, ignore_index=True)
    if len(sample_df) > sample_size:
        sample_df = sample_df.sample(n=sample_size, random_state=random_state)

    return sample_df.sort_values("tweet_id").reset_index(drop=True)


sample_df = build_representative_sample(
    dataframe=customer_tickets,
    target_size=TARGET_SAMPLE_SIZE,
    random_state=SEED,
)

sample_output_path = data_sample_dir / "customer_support_sample.csv"
sample_df.to_csv(sample_output_path, index=False)

print(f"Saved representative sample to: {sample_output_path}")
print(f"Sample shape: {sample_df.shape}")
print(
    "Brand coverage in sample:",
    sample_df["brand_hint"].nunique(),
)

sample_summary = pd.DataFrame(
    {
        "full_dataset_count": customer_tickets["brand_hint"].value_counts(),
        "sample_count": sample_df["brand_hint"].value_counts(),
    }
).fillna(0).astype(int)
sample_summary["sample_share"] = (
    sample_summary["sample_count"] / sample_summary["sample_count"].sum()
).round(4)

display(sample_summary.head(15))
display(sample_df[["tweet_id", "author_id", "brand_hint", "text"]].head(10))


Saved representative sample to: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\data\sample\customer_support_sample.csv
Sample shape: (49, 12)
Brand coverage in sample: 16


,full_dataset_count,sample_count,sample_share
brand_hint,,,
AppleSupport,13,13,0.2653
SpotifyCares,8,8,0.1633
Tesco,7,7,0.1429
unknown,6,6,0.1224
VirginTrains,3,3,0.0612
British_Airways,2,2,0.0408
ChaseSupport,1,1,0.0204
105837,1,1,0.0204
O2,1,1,0.0204


,tweet_id,author_id,brand_hint,text
0,119237,105834,unknown,@AppleSupport causing the reply to be disregarded and the tapped notification under the keyboard is opened😡😡😡
1,119239,105835,ChaseSupport,@76328 I really hope you all change but I'm sure you won't! Because you don't have to!
2,119241,105836,VirginTrains,@VirginTrains see attached error message. I've tried leaving a voicemail several times in the past week https://t.co/NxVZjlYx1k
3,119242,105836,VirginTrains,@VirginTrains I still haven't heard &amp; the number I'm directed to by phone is a dead end &amp; the live chat doesn't work. Can someone call me?
4,119244,105836,VirginTrains,"@VirginTrains yep, I've tried laptop too several times over the past week and again today. I've tried different browsers too"
5,119249,105837,AppleSupport,"@105838 @AppleSupport Me too am suffering , hope the can find a solution"
6,119250,105838,105837,"@AppleSupport hi #apple, I’ve a concern about the latest ios is too slow on #iphone6 and i am not happy with it. Any solution please?"
7,119253,105839,AppleSupport,I just updated my phone and suddenly everything takes ages to load wtf @76099 this update sux I hate it fix it bye
8,119255,105840,SpotifyCares,@SpotifyCares Thanks! Version 8.4.22.857 armv7 on anker bluetooth speaker on Samsung Galaxy Tab A (2016) Model SM-T280 Does distance from speaker matter?
9,119256,105840,SpotifyCares,@76495 @91226 Please help! Spotify Premium skipping through songs constantly on android tablet &amp; bluetooth speaker. Tried everything!


## 5. Weak Labeling Rule

In [5]:
from backend.app.services.features import extract_features, weak_label_priority

sample_df["priority_label"] = sample_df["text"].apply(weak_label_priority)
label_distribution = sample_df["priority_label"].value_counts().rename_axis("priority").reset_index(name="count")
label_distribution["share"] = (label_distribution["count"] / len(sample_df)).round(3)

print("Weak-label distribution")
display(label_distribution)

display(
    sample_df[["tweet_id", "brand_hint", "text", "priority_label"]]
    .sort_values("priority_label")
    .head(12)
)


Weak-label distribution


,priority,count,share
0,low,33,0.673
1,normal,15,0.306
2,high,1,0.020


,tweet_id,brand_hint,text,priority_label
12,119263,AppleSupport,"@AppleSupport after the 11.0.2 my phone just sucks most of the apps are broken, wifi disconnects frequently #apple #ios1102 #painfulupdate",high
0,119237,unknown,@AppleSupport causing the reply to be disregarded and the tapped notification under the keyboard is opened😡😡😡,low
23,119285,SpotifyCares,@SpotifyCares problem has come back again today. is there some kind of bug?,low
47,119331,105859,They reschedule my shit for tomorrow https://t.co/RsvZcT982t,low
27,119292,AppleSupport,@AppleSupport #ios11update - is still killing my battery within 12 hours - phone is 10 months old - it’s a disgrace - used to get 2 days,low
28,119294,AppleSupport,"Took my phone off charge at 7:20am.\n\n8:03am - 60% battery remaining.\n\n@76099 plz I beg you, sort your battery life out😩",low
32,119301,AppleSupport,@76099 @AppleSupport fix this update. It’s horrible,low
33,119302,British_Airways,"@British_Airways Thanks for your answer. Response onsite was that Aircraft will not depart today, so rebooking to @105853 with now 4h delay...",low
34,119305,unknown,@Ask_Spectrum It's just in my yard. I've called 4-5 times in 6 weeks. I'm not a customer but it's pretty tempting. I do have some coaxial,low
35,119306,Ask_Spectrum,@76501 I guess this means free cable for the neighborhood. https://t.co/FBgEE9YXPm,low


## 6. Feature Engineering

In [6]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

feature_rows = [extract_features(text).__dict__ for text in sample_df["text"]]
engineered_df = pd.DataFrame(feature_rows)

vectorizer = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    min_df=1,
    stop_words="english",
)
tfidf_matrix = vectorizer.fit_transform(sample_df["text"])
engineered_matrix = engineered_df.to_numpy(dtype=float)
combined_matrix = hstack([tfidf_matrix, engineered_matrix])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(sample_df["priority_label"])

print("TF-IDF shape:", tfidf_matrix.shape)
print("Engineered feature shape:", engineered_matrix.shape)
display(engineered_df.describe().T)


TF-IDF shape: (49, 775)
Engineered feature shape: (49, 10)


,count,mean,std,min,25%,50%,75%,max
char_count,49.0,106.000000,35.327633,32.000000,78.000000,108.000000,137.000000,170.000000
word_count,49.0,18.081633,6.617013,3.000000,14.000000,19.000000,22.000000,30.000000
exclamation_count,49.0,0.346939,0.925361,0.000000,0.000000,0.000000,0.000000,5.000000
question_count,49.0,0.285714,0.540062,0.000000,0.000000,0.000000,0.000000,2.000000
uppercase_ratio,49.0,0.060816,0.037027,0.014925,0.033333,0.050847,0.082192,0.188679
urgent_keyword_count,49.0,0.224490,0.421570,0.000000,0.000000,0.000000,0.000000,1.000000
high_impact_keyword_count,49.0,0.285714,0.456435,0.000000,0.000000,0.000000,1.000000,1.000000
negative_keyword_count,49.0,0.061224,0.242226,0.000000,0.000000,0.000000,0.000000,1.000000
has_url,49.0,0.122449,0.331201,0.000000,0.000000,0.000000,0.000000,1.000000
has_mention,49.0,0.979592,0.142857,0.000000,1.000000,1.000000,1.000000,1.000000


## 7. Model Comparison

In [7]:
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

feature_sets = {
    "tfidf_only": tfidf_matrix,
    "engineered_only": engineered_matrix,
    "tfidf_engineered": combined_matrix,
}
models = {
    "LogisticRegression": LogisticRegression(max_iter=3000, class_weight="balanced"),
    "LinearSVC": LinearSVC(max_iter=20000, class_weight="balanced"),
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=1,
    ),
}

results = []
trained_models = {}

class_counts = pd.Series(y).value_counts()
stratify_labels = y if class_counts.min() >= 2 else None
if stratify_labels is None:
    print(
        "Warning: at least one class has fewer than 2 rows, so this smoke-test "
        "run uses a regular split. Use twcs.csv/10k sample for final metrics."
    )

for feature_name, features in feature_sets.items():
    x_train, x_test, y_train, y_test = train_test_split(
        features,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=stratify_labels,
    )
    for model_name, model in models.items():
        start = time.perf_counter()
        model.fit(x_train, y_train)
        predictions = model.predict(x_test)
        latency_ms = ((time.perf_counter() - start) / max(len(y_test), 1)) * 1000
        macro_f1 = f1_score(y_test, predictions, average="macro")
        results.append(
            {
                "feature_set": feature_name,
                "model": model_name,
                "accuracy": accuracy_score(y_test, predictions),
                "macro_f1": macro_f1,
                "latency_ms_per_prediction": latency_ms,
            }
        )
        trained_models[(feature_name, model_name)] = (model, x_test, y_test, predictions)

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(results_df)

best_row = results_df.loc[results_df["feature_set"].eq("tfidf_engineered")].iloc[0]
best_key = (best_row["feature_set"], best_row["model"])
best_model, x_test, y_test, predictions = trained_models[best_key]

print("Selected model:", best_key)
print(classification_report(
    y_test,
    predictions,
    labels=list(range(len(label_encoder.classes_))),
    target_names=label_encoder.classes_,
    zero_division=0,
))

display(pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=list(range(len(label_encoder.classes_)))),
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
))


E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\backend\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,feature_set,model,accuracy,macro_f1,latency_ms_per_prediction
3,engineered_only,LogisticRegression,1.0,1.000000,19.72878
7,tfidf_engineered,LinearSVC,1.0,1.000000,9.49460
6,tfidf_engineered,LogisticRegression,0.9,0.898990,51.38616
5,engineered_only,RandomForestClassifier,0.9,0.898990,27.31331
4,engineered_only,LinearSVC,0.9,0.898990,0.16871
0,tfidf_only,LogisticRegression,0.5,0.333333,0.95102
2,tfidf_only,RandomForestClassifier,0.5,0.333333,31.29389
1,tfidf_only,LinearSVC,0.5,0.333333,0.17152
8,tfidf_engineered,RandomForestClassifier,0.5,0.333333,30.70875


Selected model: ('tfidf_engineered', 'LinearSVC')
              precision    recall  f1-score   support

        high       0.00      0.00      0.00         0
         low       1.00      1.00      1.00         5
      normal       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       0.67      0.67      0.67        10
weighted avg       1.00      1.00      1.00        10



,high,low,normal
high,0,0,0
low,0,5,0
normal,0,0,5


## 8. Error Analysis

In [8]:
error_cases = sample_df.iloc[getattr(x_test, "indices", np.arange(min(len(sample_df), len(y_test))))[:0]].copy()
comparison_df = pd.DataFrame(
    {
        "actual": label_encoder.inverse_transform(y_test),
        "predicted": label_encoder.inverse_transform(predictions),
    }
)
comparison_df["is_correct"] = comparison_df["actual"].eq(comparison_df["predicted"])

print("Prediction correctness summary")
display(comparison_df["is_correct"].value_counts().rename_axis("correct").reset_index(name="count"))
print("Important limitation: labels are weak supervision from rules, not human annotations.")


Prediction correctness summary


,correct,count
0,True,10


Important limitation: labels are weak supervision from rules, not human annotations.


## 9. Export Artifacts

In [9]:
import json
import joblib

artifacts_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, artifacts_dir / "priority_model.joblib")
joblib.dump(vectorizer, artifacts_dir / "vectorizer.joblib")
joblib.dump(label_encoder, artifacts_dir / "label_encoder.joblib")

metadata = {
    "selected_feature_set": best_key[0],
    "selected_model": best_key[1],
    "rows": int(len(sample_df)),
    "label_distribution": sample_df["priority_label"].value_counts().to_dict(),
    "model_comparison": results_df.to_dict(orient="records"),
    "recommendation": (
        "Use the trained ML model for high-volume routing because latency and cost are much lower; "
        "use the LLM for low-confidence or high-risk cases where reasoning is valuable."
    ),
}
(artifacts_dir / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Exported artifacts to:", artifacts_dir)
print(json.dumps(metadata, indent=2)[:1500])


Exported artifacts to: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\artifacts
{
  "selected_feature_set": "tfidf_engineered",
  "selected_model": "LinearSVC",
  "rows": 49,
  "label_distribution": {
    "low": 33,
    "normal": 15,
    "high": 1
  },
  "model_comparison": [
    {
      "feature_set": "engineered_only",
      "model": "LogisticRegression",
      "accuracy": 1.0,
      "macro_f1": 1.0,
      "latency_ms_per_prediction": 19.728779999968538
    },
    {
      "feature_set": "tfidf_engineered",
      "model": "LinearSVC",
      "accuracy": 1.0,
      "macro_f1": 1.0,
      "latency_ms_per_prediction": 9.494600000016362
    },
    {
      "feature_set": "tfidf_engineered",
      "model": "LogisticRegression",
      "accuracy": 0.9,
      "macro_f1": 0.898989898989899,
      "latency_ms_per_prediction": 51.38615999994727
    },
    {
      "feature_set": "engineered_only",
      "model": "RandomForestClassifier",
      "accuracy": 0.9,
      